In [1]:
# =============================================================================
# CS-781 Phase 2 — L-NER on Arch #1 (word-level tagging + Viterbi)
# RUN-ALL VERSION: trains every technique one after another, then zips each separately.
#
#   full      full fine-tuning
#   lorma     LoRMA+  (multiplicative low-rank on Wq, Wv)       [compulsory]
#   lora      LoRA    (additive low-rank on Wq, Wv)
#   adapter   Houlsby-style bottleneck adapters after attention and FFN
#   partial   fine-tune only the top PARTIAL_LAYERS blocks + final norm
#   normtune  train only the RMSNorm gains (BitFit-style sparse method)
#   kd        knowledge distillation: 6-layer student learns from the fully fine-tuned teacher
#
# Kaggle: paste each "# %%" block as its own cell, in order. Only edit MODEL_CKPT_PATH
# (cell 2), then "Save Version -> Save & Run All".
#
# Outputs:
#   /kaggle/working/cs781_phase2_arch1_<tech>_s<seed>/   (ner_best.pt, prediction.json, pkl log-probs, ...)
#   /kaggle/working/submissions/submission_phase2_arch1_<tech>_s<seed>.zip   (Codabench upload)
#   /kaggle/working/results_zips/results_<tech>_s<seed>.zip                  (everything incl. ner_best.pt)
# =============================================================================

# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, re, itertools, pickle
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# %% [2] Config
TECHNIQUES = ["full", "lorma", "lora", "adapter", "partial", "normtune", "kd"]   # keep "full" before "kd" (kd reuses its teacher)
SEED = 42                     # training randomness (head init, dropout, batch order)
SPLIT_SEED = 42               # decides the held-out docs; keep fixed so runs can be ensembled
SKIP_DONE = True              # if a technique already finished (files exist), skip it -> safe to re-run after a crash
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2token47"          # tokenizer
# !!! EDIT: the Arch #1 retrain checkpoint (model_best.pt), same one used before
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2model47/model_best.pt"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_TEXT_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold2-text.json"]
TRAIN_GOLD_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold2-gold.json"]
LABELS_FILE = "Phase_2_LNER/Train/labels.txt"
TEST_TEXT_FILE = "Phase_2_LNER/Test/fold3-text.json"

HOLDOUT_DOCS = 10             # 0 = final run on all 70 docs (use the epoch count found with holdout)
WORD_POOL = "last"
STRIDE = 512
BATCH_SIZE = 8
EVAL_BATCH = 8
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
DROPOUT_FT = 0.1
O_WEIGHT = 1.0

LORMA_R, LORMA_ALPHA = 8, 8           # also used for LoRA
ADAPTER_DIM = 32
PARTIAL_LAYERS = 4
STUDENT_KEEP = [1, 3, 5, 7, 9, 11]    # KD student keeps these Phase-1 layers (6 of 12)
KD_T, KD_ALPHA = 2.0, 0.5             # temperature and weight of the distillation term
TEACHER_NER_PATH = None               # None = KD automatically uses the "full" run's checkpoint from this notebook

HP = {   # starting points, not tuned
    "full":     dict(epochs=6,  lr_body=5e-5, lr_head=5e-4),
    "lorma":    dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "lora":     dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "adapter":  dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "partial":  dict(epochs=8,  lr_body=1e-4, lr_head=5e-4),
    "normtune": dict(epochs=20, lr_body=1e-3, lr_head=1e-3),
    "kd":       dict(epochs=8,  lr_body=1e-4, lr_head=5e-4),
}
for _t in TECHNIQUES:
    assert _t in HP, _t
if "kd" in TECHNIQUES and "full" in TECHNIQUES:
    assert TECHNIQUES.index("full") < TECHNIQUES.index("kd"), "put 'full' before 'kd'"

ROOT_OUT = "/kaggle/working"
def out_dir_for(tech):
    return f"{ROOT_OUT}/cs781_phase2_arch1_{tech}_s{SEED}"
print("Will run:", TECHNIQUES, "| seed:", SEED)

# %% [3] Load Phase-2 data
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

LABELS = [x.strip() for x in open(hf_local(LABELS_FILE), encoding="utf-8") if x.strip()]
assert len(LABELS) == 12, LABELS

ID2TAG = {0: ("O", None)}
for i, lab in enumerate(LABELS):
    ID2TAG[1 + 2 * i] = ("B", lab)
    ID2TAG[2 + 2 * i] = ("I", lab)
NUM_TAGS = len(ID2TAG)
B_ID = {lab: 1 + 2 * i for i, lab in enumerate(LABELS)}
I_ID = {lab: 2 + 2 * i for i, lab in enumerate(LABELS)}

def norm_entity(e):
    if isinstance(e, (list, tuple)):
        s, en, lab = e[0], e[1], e[2]
        txt = e[3] if len(e) > 3 else None
    elif isinstance(e, dict):
        def pick(*ks):
            for k in ks:
                if k in e:
                    return e[k]
            return None
        s = pick("start", "start_char", "start_offset", "begin")
        en = pick("end", "end_char", "end_offset")
        lab = pick("label", "type", "entity", "tag")
        txt = pick("text", "entity_text", "span")
    else:
        raise ValueError(f"Unrecognised entity format: {e!r}")
    return int(s), int(en), str(lab), txt

texts, gold = {}, {}
for tf, gf in zip(TRAIN_TEXT_FILES, TRAIN_GOLD_FILES):
    t = json.load(open(hf_local(tf), encoding="utf-8"))
    g = json.load(open(hf_local(gf), encoding="utf-8"))
    for k, v in t.items():
        assert k not in texts, f"duplicate doc id {k}"
        texts[k] = v
    for k, v in g.items():
        if isinstance(v, dict):
            v = v.get("entities") or v.get("annotations") or v.get("labels") or []
        gold[k] = [norm_entity(e) for e in v]
for k in texts:
    gold.setdefault(k, [])
lab_counts = Counter(l for ents in gold.values() for (_, _, l, _) in ents)
print("Docs:", len(texts), "| entities per label:", dict(lab_counts))

# %% [4] Tokenizer (Phase-1 BPE) + tokenization with char offsets AND word ids
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
EOW = "\u0100"
WORD_RE = re.compile(r"\S+")

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

def tokenize_with_offsets(text):
    ids, offs, wid, words = [], [], [], []
    for wi, m in enumerate(WORD_RE.finditer(text)):
        word, w0 = m.group(), m.start()
        words.append((w0, m.end()))
        b2c = []
        for ci, ch in enumerate(word):
            for _ in ch.encode("utf-8", "ignore"):
                b2c.append(ci)
        bpos = 0
        for sym in bpe_encode_word(word):
            nb = sum(1 for ch in sym if ch != EOW)
            if nb == 0 or not b2c:
                ce = w0 + len(word)
                offs.append((ce, ce))
            else:
                sb = min(bpos, len(b2c) - 1)
                eb = min(bpos + nb, len(b2c))
                offs.append((w0 + b2c[sb], w0 + b2c[eb - 1] + 1))
            bpos += nb
            ids.append(token_to_id.get(sym, UNK_ID))
            wid.append(wi)
    return ids, offs, wid, words

# %% [5] Word-level records and labels
def encode_doc(text, spans=None):
    ids, offs, wid, words = tokenize_with_offsets(text)
    nz_by_word = [[] for _ in words]
    for t, (a, b) in enumerate(offs):
        if b > a:
            nz_by_word[wid[t]].append(t)
    keep = [k for k, lst in enumerate(nz_by_word) if lst]
    words_v = [words[k] for k in keep]
    pool = [nz_by_word[k][-1] if WORD_POOL == "last" else nz_by_word[k][0] for k in keep]
    rec = {"ids": ids, "words": words_v, "pool": pool, "len": len(text)}
    if spans is None:
        return rec
    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    char_sid = [-1] * len(text)
    for k, (s, e, lab, _) in enumerate(spans):
        for c in range(max(0, s), min(len(text), e)):
            char_sid[c] = k
    tok_labels = [-100] * len(ids)
    span_rng = {}
    prev_sid = -1
    for k, (ws, we) in enumerate(words_v):
        sid = -1
        for c in range(ws, we):
            if char_sid[c] >= 0:
                sid = char_sid[c]
                break
        if sid < 0:
            lab_id = 0
        else:
            lab = spans[sid][2]
            lab_id = B_ID[lab] if sid != prev_sid else I_ID[lab]
            lo, hi = span_rng.get(sid, (ws, we))
            span_rng[sid] = (min(lo, ws), max(hi, we))
        tok_labels[pool[k]] = lab_id
        prev_sid = sid
    rec.update(labels=tok_labels, spans=spans, span_rng=span_rng)
    return rec

recs = {k: encode_doc(texts[k], gold[k]) for k in texts}
print(f"Encoded {len(recs)} docs | tokens: {sum(len(r['ids']) for r in recs.values()):,} | "
      f"words: {sum(len(r['words']) for r in recs.values()):,}")

# %% [6] Word-level oracle -> punctuation trim rule
def trim_span(text, s, e, lead, trail):
    while s < e and (text[s].isspace() or text[s] in lead):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in trail):
        e -= 1
    return s, e

LEAD_OPTS = ["", "(\"'[\u201c\u2018"]
TRAIL_OPTS = ["", ",;", ",;:", ",;:.", ",;:.)\"'\u201d\u2019]", ",;:)\"'\u201d\u2019]"]
total_spans = sum(len(r["spans"]) for r in recs.values())
results = []
for lead, trail in itertools.product(LEAD_OPTS, TRAIL_OPTS):
    ok = 0
    for k, r in recs.items():
        for sid, (ws, we) in r["span_rng"].items():
            s, e = r["spans"][sid][0], r["spans"][sid][1]
            ok += int(trim_span(texts[k], ws, we, lead, trail) == (s, e))
    results.append((ok / max(1, total_spans), lead, trail))
results.sort(key=lambda x: -x[0])
ORACLE_RATE, TRIM_LEAD, TRIM_TRAIL = results[0]
print(f"Word-level oracle ceiling: {ORACLE_RATE:.4f} | TRIM_LEAD={TRIM_LEAD!r} TRIM_TRAIL={TRIM_TRAIL!r}")

# %% [7] Model: Arch #1 backbone + PEFT modules + tag head
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRMAPlus(nn.Module):
    """h' = h + (alpha/r) B(A h) = ((alpha/r) B A + I) W0 x   with h = W0 x   (multiplicative, B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, h):
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class LoRAAdapter(nn.Module):
    """Additive update (alpha/r) B(A x), added to W0 x   (LoRA, B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * F.linear(F.linear(x, self.A), self.B)

class BottleneckAdapter(nn.Module):
    """Houlsby-style adapter: h + up(gelu(down(h))); `up` starts at zero, so it begins as the identity."""
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim, self.n_embd = n_head, n_embd // n_head, n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.adapt_mode = None     # "lorma" | "lora" | None
        self.q_ad = None
        self.v_ad = None
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        if self.adapt_mode == "lorma":
            q, v = self.q_ad(q), self.v_ad(v)
        elif self.adapt_mode == "lora":
            q, v = q + self.q_ad(x), v + self.v_ad(x)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
        self.ad1 = None            # Houlsby adapters (after attention / after FFN)
        self.ad2 = None
    def forward(self, x, cos, sin):
        a = self.attn(self.ln1(x), cos, sin)
        if self.ad1 is not None:
            a = self.ad1(a)
        x = x + a
        f = self.ffn(self.ln2(x))
        if self.ad2 is not None:
            f = self.ad2(f)
        return x + f

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)

class NERModel(nn.Module):
    def __init__(self, backbone, n_embd, num_tags, dropout):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_embd, num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, idx):
        return self.classifier(self.drop(self.backbone.forward_hidden(idx)))

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
CONTENT = cfg["block_size"] - 1

def make_student_state(sd, keep):
    """Keep only the Phase-1 layers listed in `keep` (renumbered 0..len-1); copy everything else."""
    new = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            parts = k.split(".")
            old = int(parts[1])
            if old in keep:
                parts[1] = str(keep.index(old))
                new[".".join(parts)] = v
        else:
            new[k] = v
    return new

def build_ner(technique, keep_layers=None):
    sd, n_layer = ckpt["model_state_dict"], cfg["n_layer"]
    if keep_layers is not None:
        sd, n_layer = make_student_state(sd, keep_layers), len(keep_layers)
    bb = LegalLLaMA(cfg["vocab_size"], n_layer, cfg["n_head"], cfg["n_embd"],
                    cfg["block_size"], cfg["ffn_mult"], DROPOUT_FT)
    bb.load_state_dict(sd)
    d = cfg["n_embd"]
    if technique in ("lorma", "lora", "adapter", "partial", "normtune"):
        for p in bb.parameters():
            p.requires_grad = False                         # freeze W0 first; new modules below are trainable
        if technique in ("lorma", "lora"):
            for blk in bb.blocks:
                blk.attn.adapt_mode = technique
                make = LoRMAPlus if technique == "lorma" else LoRAAdapter
                blk.attn.q_ad, blk.attn.v_ad = make(d, LORMA_R, LORMA_ALPHA), make(d, LORMA_R, LORMA_ALPHA)
        elif technique == "adapter":
            for blk in bb.blocks:
                blk.ad1, blk.ad2 = BottleneckAdapter(d, ADAPTER_DIM), BottleneckAdapter(d, ADAPTER_DIM)
        elif technique == "partial":
            for blk in bb.blocks[-PARTIAL_LAYERS:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in bb.ln_f.parameters():
                p.requires_grad = True
        elif technique == "normtune":
            for m in bb.modules():
                if isinstance(m, RMSNorm):
                    m.weight.requires_grad = True
    net = NERModel(bb, d, NUM_TAGS, DROPOUT_FT).to(device)
    n_tot = sum(p.numel() for p in net.parameters())
    n_trn = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"[{technique}{' student' if keep_layers else ''}] trainable {n_trn/1e6:.3f}M of {n_tot/1e6:.1f}M params ({100*n_trn/n_tot:.2f}%)")
    return net

# %% [8] Windows, datasets, split
def make_windows(n, content, stride):
    if n <= content:
        return [(0, n)]
    starts = list(range(0, n - content + 1, stride))
    if starts[-1] + content < n:
        starts.append(n - content)
    return [(s, s + content) for s in starts]

doc_ids = sorted(recs.keys())
random.Random(SPLIT_SEED).shuffle(doc_ids)
val_ids = doc_ids[:HOLDOUT_DOCS]
train_ids = doc_ids[HOLDOUT_DOCS:]
print(f"Train docs: {len(train_ids)} | held-out docs: {len(val_ids)}")

class WindowDataset(Dataset):
    def __init__(self, doc_keys):
        self.items = []
        for k in doc_keys:
            r = recs[k]
            for (s, e) in make_windows(len(r["ids"]), CONTENT, STRIDE):
                x = [BOS_ID] + r["ids"][s:e]
                y = [-100] + r["labels"][s:e]
                pad = CONTENT + 1 - len(x)
                self.items.append((x + [PAD_ID] * pad, y + [-100] * pad))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        x, y = self.items[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_ds = WindowDataset(train_ids)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
print("Training windows:", len(train_ds), "| steps/epoch:", len(train_loader))

# %% [9] Inference, word-level Viterbi, strict macro-F1, error analysis
@torch.no_grad()
def predict_logp(net, rec):
    net.eval()
    n = len(rec["ids"])
    wins = make_windows(n, CONTENT, STRIDE)
    store = torch.zeros(n, NUM_TAGS)
    best = torch.full((n,), -1, dtype=torch.long)
    for i in range(0, len(wins), EVAL_BATCH):
        chunk = wins[i:i + EVAL_BATCH]
        batch = torch.full((len(chunk), CONTENT + 1), PAD_ID, dtype=torch.long)
        for r_i, (s, e) in enumerate(chunk):
            batch[r_i, 0] = BOS_ID
            batch[r_i, 1:1 + e - s] = torch.tensor(rec["ids"][s:e], dtype=torch.long)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = net(batch.to(device))
        logp = F.log_softmax(logits.float(), dim=-1).cpu()
        for r_i, (s, e) in enumerate(chunk):
            L = e - s
            pos = torch.arange(L)
            idx = s + pos
            upd = pos > best[idx]
            store[idx[upd]] = logp[r_i, 1:1 + L][upd]
            best[idx[upd]] = pos[upd]
    return store

def build_transitions():
    NEG = -1e9
    trans = np.zeros((NUM_TAGS, NUM_TAGS), dtype=np.float32)
    for a in range(NUM_TAGS):
        for b in range(NUM_TAGS):
            tag_a, ent_a = ID2TAG[a]
            tag_b, ent_b = ID2TAG[b]
            if tag_b == "I" and not (tag_a in ("B", "I") and ent_a == ent_b):
                trans[a, b] = NEG
    start = np.array([NEG if ID2TAG[b][0] == "I" else 0.0 for b in range(NUM_TAGS)], dtype=np.float32)
    return trans, start
TRANS, START = build_transitions()

def viterbi(logp):
    n = logp.shape[0]
    score = START + logp[0]
    back = np.zeros((n, NUM_TAGS), dtype=np.int64)
    for t in range(1, n):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = score.argmax()
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

def word_tags(logp, rec, mode):
    if not rec["pool"]:
        return np.zeros(0, dtype=np.int64)
    lp = logp[rec["pool"]].numpy()
    return lp.argmax(1) if mode == "argmax" else viterbi(lp)

def decode_spans(text, wtags, rec):
    spans, cur = [], None
    for k, tid in enumerate(wtags):
        ws, we = rec["words"][k]
        tid = int(tid)
        if tid == 0:
            if cur:
                spans.append(cur); cur = None
            continue
        tag, ent = ID2TAG[tid]
        if cur is not None and tag == "I" and cur[2] == ent:
            cur[1] = we
        else:
            if cur:
                spans.append(cur)
            cur = [ws, we, ent]
    if cur:
        spans.append(cur)
    out = set()
    for s, e, ent in spans:
        s2, e2 = trim_span(text, s, e, TRIM_LEAD, TRIM_TRAIL)
        if e2 > s2:
            out.add((s2, e2, ent))
    return sorted(out)

def strict_macro(truth, pred):
    stats = {l: [0, 0, 0] for l in LABELS}
    for d in truth:
        g, p = set(truth[d]), set(pred.get(d, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}
            ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    P, R, F_, per = [], [], [], {}
    for l in LABELS:
        tp, fp, fn = stats[l]
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        P.append(pr); R.append(rc); F_.append(f); per[l] = (pr, rc, f)
    return sum(F_) / len(F_), sum(P) / len(P), sum(R) / len(R), per

def evaluate_heldout(net, keys):
    truth = {k: sorted({(s, e, l) for (s, e, l, _) in gold[k]}) for k in keys}
    preds = {"argmax": {}, "viterbi": {}}
    for k in keys:
        logp = predict_logp(net, recs[k])
        for mode in preds:
            preds[mode][k] = decode_spans(texts[k], word_tags(logp, recs[k], mode), recs[k])
    return {m: strict_macro(truth, preds[m]) for m in preds}, truth, preds

def error_analysis(truth, pred):
    cats = {l: Counter() for l in LABELS}
    spurious = Counter()
    for d in truth:
        g, p = truth[d], pred[d]
        pset, gset = set(p), set(g)
        for (s, e, l) in g:
            if (s, e, l) in pset:
                cats[l]["correct"] += 1
            elif any(pl == l and ps < e and pe > s for ps, pe, pl in p):
                cats[l]["boundary"] += 1
            elif any(pl != l and ps < e and pe > s for ps, pe, pl in p):
                cats[l]["label_confusion"] += 1
            else:
                cats[l]["missed"] += 1
        for (ps, pe, pl) in p:
            if (ps, pe, pl) not in gset and not any(gs < pe and ge > ps for gs, ge, gl in g):
                spurious[pl] += 1
    print(f"{'label':10s} {'gold':>5s} {'correct':>8s} {'boundary':>9s} {'confused':>9s} {'missed':>7s} {'spurious':>9s}")
    for l in LABELS:
        c = cats[l]
        print(f"{l:10s} {sum(c.values()):5d} {c['correct']:8d} {c['boundary']:9d} {c['label_confusion']:9d} {c['missed']:7d} {spurious[l]:9d}")

# %% [10] Generic training function (optionally with a teacher for distillation)
class_w = torch.ones(NUM_TAGS)
class_w[0] = O_WEIGHT
class_w = class_w.to(device)

def fit(model, epochs, lr_body, lr_head, best_path, teacher=None):
    head_params = list(model.classifier.parameters())
    head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    trainable = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": head_params, "lr": lr_head, "base": lr_head}]
    if body_params:
        groups.insert(0, {"params": body_params, "lr": lr_body, "base": lr_body})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    total_steps = epochs * len(train_loader)
    warm = max(1, int(WARMUP_FRAC * total_steps))
    def lr_factor(step):
        if step < warm:
            return step / warm
        prog = (step - warm) / max(1, total_steps - warm)
        return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))

    best_f1, step, t0, history = -1.0, 0, time.time(), []
    if teacher is not None:
        teacher.eval()
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, n_b = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            for g in optimizer.param_groups:
                g["lr"] = g["base"] * lr_factor(step)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(x)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(x)
            logits = logits.float()
            loss = F.cross_entropy(logits.view(-1, NUM_TAGS), y.view(-1), weight=class_w, ignore_index=-100)
            if teacher is not None:
                mask = y != -100
                if mask.any():
                    s = F.log_softmax(logits[mask] / KD_T, dim=-1)
                    t = F.softmax(t_logits.float()[mask] / KD_T, dim=-1)
                    kd = F.kl_div(s, t, reduction="batchmean") * (KD_T ** 2)
                    loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kd
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item(); n_b += 1; step += 1
        rec_h = {"epoch": epoch, "train_loss": run_loss / max(1, n_b)}
        if val_ids:
            out, _, _ = evaluate_heldout(model, val_ids)
            rec_h.update(f1_argmax=out["argmax"][0], f1_viterbi=out["viterbi"][0])
            print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} | held-out strict macro-F1: "
                  f"argmax {out['argmax'][0]:.4f} | viterbi {out['viterbi'][0]:.4f} "
                  f"(P {out['viterbi'][1]:.4f} R {out['viterbi'][2]:.4f}) | {(time.time()-t0)/60:.1f}m")
            if out["viterbi"][0] > best_f1:
                best_f1 = out["viterbi"][0]
                torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": best_f1}, best_path)
                print("  -> new best checkpoint saved")
        else:
            print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} (no held-out; final run on all docs)")
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": None}, best_path)
        history.append(rec_h)
    model.load_state_dict(torch.load(best_path, map_location="cpu")["model"])
    return history

# %% [11] Run ONE technique end-to-end (train -> eval -> save log-probs -> predict test)
import gc, traceback

def set_seed(s):
    random.seed(s); torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def run_technique(tech):
    out_dir = out_dir_for(tech)
    os.makedirs(out_dir, exist_ok=True)
    best_path = os.path.join(out_dir, "ner_best.pt")
    pred_path = os.path.join(out_dir, "prediction.json")
    summ_path = os.path.join(out_dir, "summary.json")

    # ---- resume support
    done_files = [pred_path, summ_path, os.path.join(out_dir, "word_logp_val.pkl"),
                  os.path.join(out_dir, "word_logp_test.pkl")]
    if SKIP_DONE and all(os.path.exists(p) for p in done_files):
        print(f"\n##### [{tech}] already finished -> skipping (set SKIP_DONE=False to redo)")
        return json.load(open(summ_path))

    print(f"\n{'='*78}\n##### TECHNIQUE: {tech} | seed {SEED} | {HP[tech]}\n{'='*78}")
    set_seed(SEED)
    t_run = time.time()
    teacher = None

    # ---- train
    if tech == "kd":
        teacher = build_ner("full")
        full_ckpt = os.path.join(out_dir_for("full"), "ner_best.pt")
        if TEACHER_NER_PATH and os.path.exists(TEACHER_NER_PATH):
            teacher_src = TEACHER_NER_PATH
        elif os.path.exists(full_ckpt):
            teacher_src = full_ckpt
        else:
            teacher_src = None
        if teacher_src:
            teacher.load_state_dict(torch.load(teacher_src, map_location="cpu")["model"])
            print("Loaded teacher from", teacher_src)
        else:
            print("No saved teacher found -> training the teacher (full fine-tuning) first")
            fit(teacher, best_path=os.path.join(out_dir, "teacher_best.pt"), **HP["full"])
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        if val_ids:
            out_t, _, _ = evaluate_heldout(teacher, val_ids)
            print(f"TEACHER held-out strict macro-F1 (viterbi): {out_t['viterbi'][0]:.4f}")
        model = build_ner("full", keep_layers=STUDENT_KEEP)
        t_params = sum(p.numel() for p in teacher.parameters())
        s_params = sum(p.numel() for p in model.parameters())
        print(f"Teacher {t_params/1e6:.1f}M -> student {s_params/1e6:.1f}M params ({100*(1-s_params/t_params):.0f}% smaller)")
        history = fit(model, best_path=best_path, teacher=teacher, **HP["kd"])
    else:
        model = build_ner(tech)
        history = fit(model, best_path=best_path, **HP[tech])
    json.dump(history, open(os.path.join(out_dir, "history.json"), "w"), indent=2)
    print(f"[{tech}] training finished in {(time.time()-t_run)/60:.1f} min")

    # ---- held-out scores + error analysis + word log-probs
    decode_mode, f1_v, f1_a = "viterbi", None, None
    val_dump = {}
    if val_ids:
        out, val_truth, val_preds = evaluate_heldout(model, val_ids)
        decode_mode = max(out, key=lambda m: out[m][0])
        f1_a, f1_v = out["argmax"][0], out["viterbi"][0]
        print(f"[{tech}] argmax F1 {f1_a:.4f} | viterbi F1 {f1_v:.4f} -> using {decode_mode}")
        per = out[decode_mode][3]
        for l in LABELS:
            print(f"  {l:10s} P={per[l][0]:.3f} R={per[l][1]:.3f} F1={per[l][2]:.3f}  (gold n={lab_counts.get(l,0)})")
        print()
        error_analysis(val_truth, val_preds[decode_mode])
        for k in val_ids:
            lp = predict_logp(model, recs[k])[recs[k]["pool"]].numpy().astype(np.float16)
            val_dump[k] = {"words": recs[k]["words"], "logp": lp}

    # ---- test predictions
    test_texts = json.load(open(hf_local(TEST_TEXT_FILE), encoding="utf-8"))
    test_pred, test_dump = {}, {}
    for k, text in test_texts.items():
        rec = encode_doc(text)
        logp = predict_logp(model, rec)
        spans = decode_spans(text, word_tags(logp, rec, decode_mode), rec)
        test_pred[k] = [[s, e, l] for (s, e, l) in spans]
        test_dump[k] = {"words": rec["words"], "logp": logp[rec["pool"]].numpy().astype(np.float16)}

    assert set(test_pred) == set(test_texts), "ID mismatch"
    for k, spans in test_pred.items():
        n = len(test_texts[k])
        for s, e, l in spans:
            assert 0 <= s < e <= n and l in LABELS, (k, s, e, l)
    tot = sum(len(v) for v in test_pred.values())
    print(f"[{tech}] test docs {len(test_pred)} | predicted entities {tot} | per doc {tot/len(test_pred):.1f} (training gold ~120/doc)")

    pickle.dump({"val_ids": val_ids, "data": val_dump}, open(os.path.join(out_dir, "word_logp_val.pkl"), "wb"))
    pickle.dump({"data": test_dump}, open(os.path.join(out_dir, "word_logp_test.pkl"), "wb"))
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(test_pred, f)

    summary = {"technique": tech, "f1_viterbi": f1_v, "f1_argmax": f1_a, "decode": decode_mode,
               "test_entities": tot, "minutes": round((time.time() - t_run) / 60, 1)}
    json.dump(summary, open(summ_path, "w"), indent=2)

    del model, teacher
    return summary

# %% [12] Run ALL techniques one after another
all_results = []
for tech in TECHNIQUES:
    try:
        res = run_technique(tech)
    except Exception:
        print(f"\n!!!!! [{tech}] FAILED -- continuing with the next technique")
        traceback.print_exc()
        res = {"technique": tech, "error": True}
    all_results.append(res)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 78 + "\nSUMMARY (held-out strict macro-F1)\n" + "=" * 78)
print(f"{'technique':10s} {'viterbi':>8s} {'argmax':>8s} {'decode':>8s} {'test ents':>10s} {'min':>6s}")
for r in all_results:
    if r.get("error"):
        print(f"{r['technique']:10s}  FAILED")
        continue
    f1v = f"{r['f1_viterbi']:.4f}" if r["f1_viterbi"] is not None else "   n/a"
    f1a = f"{r['f1_argmax']:.4f}" if r["f1_argmax"] is not None else "   n/a"
    print(f"{r['technique']:10s} {f1v:>8s} {f1a:>8s} {r['decode']:>8s} {r['test_entities']:>10d} {r['minutes']:>6.1f}")
json.dump(all_results, open(f"{ROOT_OUT}/all_results_summary.json", "w"), indent=2)

# %% [13] Zip maker -- separate zips for EACH technique (standalone; works after a restart too)
import os, json, zipfile

SEED = 42
ROOT_OUT = "/kaggle/working"
TECHNIQUES = ["full", "lorma", "lora", "adapter", "partial", "normtune", "kd"]
SUB_DIR = f"{ROOT_OUT}/submissions"
RES_DIR = f"{ROOT_OUT}/results_zips"
INCLUDE_CKPT = True      # True = ner_best.pt goes inside each results zip too (set False to skip the big files)
os.makedirs(SUB_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

made = []
for t in TECHNIQUES:
    d = f"{ROOT_OUT}/cs781_phase2_arch1_{t}_s{SEED}"
    pred = f"{d}/prediction.json"
    if not os.path.exists(pred):
        print(f"[{t}] prediction.json not found -> skipped")
        continue
    n_docs = len(json.load(open(pred, encoding="utf-8")))

    # 1) Codabench submission zip (only prediction.json, at the zip root) -> upload this one
    zpath = f"{SUB_DIR}/submission_phase2_arch1_{t}_s{SEED}.zip"
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred, arcname="prediction.json")
    made.append(zpath)

    # 2) full results zip for this technique (prediction + log-probs for ensembling + history + summary [+ ckpt])
    files = ["prediction.json", "word_logp_val.pkl", "word_logp_test.pkl", "history.json", "summary.json"]
    if INCLUDE_CKPT:
        files.append("ner_best.pt")
    rpath = f"{RES_DIR}/results_{t}_s{SEED}.zip"
    with zipfile.ZipFile(rpath, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in files:
            if os.path.exists(f"{d}/{fn}"):
                zf.write(f"{d}/{fn}", arcname=fn)
    print(f"[{t}] {n_docs} docs | submission: {os.path.basename(zpath)} | results: {os.path.basename(rpath)}")

print(f"\n{len(made)} technique(s) zipped separately:")
print("  submissions ->", SUB_DIR)
print("  results     ->", RES_DIR)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda
Will run: ['full', 'lorma', 'lora', 'adapter', 'partial', 'normtune', 'kd'] | seed: 42


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Docs: 70 | entities per label: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}
Encoded 70 docs | tokens: 421,043 | words: 372,576
Word-level oracle ceiling: 0.9048 | TRIM_LEAD='("\'[“‘' TRIM_TRAIL=',;:.)"\'”’]'
Train docs: 60 | held-out docs: 10
Training windows: 658 | steps/epoch: 83

##### TECHNIQUE: full | seed 42 | {'epochs': 6, 'lr_body': 5e-05, 'lr_head': 0.0005}
[full] trainable 66.237M of 66.2M params (100.00%)
[epoch 1] loss 0.7364 | held-out strict macro-F1: argmax 0.2128 | viterbi 0.2682 (P 0.3370 R 0.2568) | 0.6m
  -> new best checkpoint saved
[epoch 2] loss 0.0873 | held-out strict macro-F1: argmax 0.2910 | viterbi 0.3742 (P 0.3837 R 0.3721) | 1.1m
  -> new best checkpoint saved
[epoch 3] loss 0.0621 | held-out strict macro-F1: argmax 0.3129 | viterbi 0.3891 (P 0.3800 R 0.4031) | 1.7m
  -> new best checkpoint saved
[epoch 4] loss 0.0495 | held-out s

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

[full] test docs 35 | predicted entities 4153 | per doc 118.7 (training gold ~120/doc)

##### TECHNIQUE: lorma | seed 42 | {'epochs': 15, 'lr_body': 0.0004, 'lr_head': 0.001}
[lorma] trainable 0.236M of 66.5M params (0.35%)
[epoch 1] loss 1.5799 | held-out strict macro-F1: argmax 0.0456 | viterbi 0.0588 (P 0.0643 R 0.0571) | 0.5m
  -> new best checkpoint saved
[epoch 2] loss 0.1390 | held-out strict macro-F1: argmax 0.2243 | viterbi 0.2757 (P 0.3094 R 0.2695) | 1.0m
  -> new best checkpoint saved
[epoch 3] loss 0.1059 | held-out strict macro-F1: argmax 0.2562 | viterbi 0.3120 (P 0.3384 R 0.3106) | 1.5m
  -> new best checkpoint saved
[epoch 4] loss 0.0944 | held-out strict macro-F1: argmax 0.2773 | viterbi 0.3631 (P 0.3573 R 0.3956) | 2.1m
  -> new best checkpoint saved
[epoch 5] loss 0.0864 | held-out strict macro-F1: argmax 0.2790 | viterbi 0.3505 (P 0.3960 R 0.3376) | 2.6m
[epoch 6] loss 0.0818 | held-out strict macro-F1: argmax 0.2896 | viterbi 0.3569 (P 0.3719 R 0.3580) | 3.1m
[epo